# 3D Visualization - Godot

If all services are not already started, open a shell and run:
```sh
python -m startup.start_all_services
```

### 1. Godot setup

1. Download and run `Godot 4.6` (the .NET version)
1. Open Godot and "Import" a project
1. Point to the `ur3e-godot` folder
1. Open the project
1. Click `Run Project`

### 2. Example loop

Run the code cell below in order to loop through example poses and view them in the 3D visualization.

In [ ]:
import random
import time
import numpy as np
from communication import protocol
from communication.rabbitmq import Rabbitmq

rmq = Rabbitmq(
    ip="localhost",
    port=5672,
    username="ur3e",
    password="ur3e",
    vhost="/",
    exchange="UR3E_AMQP",
    type="topic",
)
rmq.connect_to_server()

poses = [
    [0.0, -np.pi/2, 0, -np.pi/2, 0.0, 0.0], # point straight up
    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],         # lie down
    # [-np.pi/2, 0.0, -np.pi/2, 0.0, -np.pi/2, 0.0],
    # [0.0, -np.pi/2, 0.0, -np.pi/2, 0.0, -np.pi/2],
    # [0.0, 0.0, -np.pi/2, -np.pi/2, 0.0, 0.0],
    # [-np.pi/2, -np.pi/2, 0.0, 0.0, -np.pi/2, -np.pi/2],
]
prev_pose = []
while True:
    pose = random.choice(poses)
    msg = {
        protocol.CtrlMsgKeys.TYPE: protocol.CtrlMsgFields.LOAD_PROGRAM,
        protocol.CtrlMsgKeys.JOINT_POSITIONS: [pose],
        protocol.CtrlMsgKeys.MAX_VELOCITY:    random.randint(40, 80),
        protocol.CtrlMsgKeys.ACCELERATION:    random.randint(60, 100),
    }
    rmq.send_message(
        routing_key=protocol.ROUTING_KEY_CTRL,
        message=msg
    )
    msg = {
        protocol.CtrlMsgKeys.TYPE: protocol.CtrlMsgFields.PLAY,
    }
    rmq.send_message(
        routing_key=protocol.ROUTING_KEY_CTRL,
        message=msg
    )
    poses.remove(pose)
    if prev_pose:
        poses.append(prev_pose)
    prev_pose = pose
    time.sleep(10)